# AEGES-Q — Multiclass Intrusion Detection Model Selection

**Experiment ID:** `AEGES-Q-EXP-08`  
**Task:** Multiclass network intrusion classification  
**Dataset:** UNSW-NB15  
**Status:** Experimental model selection

## 1. Research objectives

The existing AEGES-Q classical intrusion detection system uses a Random Forest to distinguish benign traffic from malicious traffic. This experiment investigates whether a multiclass classifier can extend the system by identifying the likely attack category.

The objectives are to:

1. Audit the attack-category labels and class distribution.
2. Compare a small, complementary set of classifiers under controlled conditions.
3. Select a candidate using macro-F1, per-class recall and supporting metrics rather than accuracy alone.
4. Evaluate the selected model once on the official held-out test set.
5. Establish evidence for a subsequent production integration without modifying the existing binary detector.

## 2. Experimental protocol

- **Features:** Reuse the established AEGES-Q Variant E feature set and preprocessing.
- **Target:** `attack_cat`; exclude target and metadata columns from model inputs.
- **Model selection:** Use a stratified validation split of the official training dataset.
- **Candidates:** Dummy baseline, Logistic Regression, Random Forest and Extra Trees.
- **Primary metric:** Macro-F1, giving each attack category equal weight.
- **Secondary metrics:** Weighted-F1, accuracy, per-class precision/recall/F1, confusion matrix and training/inference time.
- **Final evaluation:** Use the official test set only after selecting the model and configuration.

All fitted preprocessing must be learned from the development training partition only. The official test set must not influence model selection.

## 3. Scope and limitations

This experiment evaluates attack-category classification on labelled benchmark data. It does not, by itself, establish real-world generalisation, identify a specific attacker, or prove that the model is ready for operational deployment. Results will guide the next engineering step.

## 4. Environment and reproducibility

This section records the execution environment, verifies the expected dataset paths and establishes the experiment's reproducibility settings before data analysis begins.

The existing preprocessing implementation is imported directly from the project so that the experiment uses the established feature definitions rather than a duplicate implementation.

In [1]:
# AEGES-Q-EXP-08: multiclass IDS model selection
# Run from the project virtual environment.

from pathlib import Path
import platform
import sys
import time
import json

import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

# Resolve the repository root whether Jupyter starts at the root
# or from inside the notebook directory.
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "08_multiclass_ids":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"Could not locate the AEGES-Q project root from {Path.cwd()}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessing import (
    REDUCED_NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    build_variant_e_preprocessor,
)

DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "UNSW_NB15_training-set.csv"
TEST_PATH = DATA_DIR / "UNSW_NB15_testing-set.csv"

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
TARGET_COLUMN = "attack_cat"

print("Experiment: AEGES-Q-EXP-08")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Project root:", PROJECT_ROOT)
print("Training dataset exists:", TRAIN_PATH.is_file())
print("Testing dataset exists:", TEST_PATH.is_file())
print("Numerical features:", len(REDUCED_NUMERICAL_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))

Experiment: AEGES-Q-EXP-08
Python: 3.13.7
Platform: Windows-11-10.0.26200-SP0
pandas: 3.0.5
NumPy: 2.5.2
scikit-learn: 1.9.0
Project root: C:\Projects\Aeges-Q
Training dataset exists: True
Testing dataset exists: True
Numerical features: 26
Categorical features: 3


## 3. Dataset Audit and Target Verification

Load the official UNSW-NB15 training and testing datasets and inspect their structure before modelling. The `attack_cat` column is the multiclass target; `label` is the existing binary target and must not be used as a model feature. The `id` column, if present, is also excluded from features.

This audit checks dataset dimensions, required columns, missing target values, attack-category naming, and class distributions. No imputation, encoding, or model fitting is performed in this step.

**Data integrity rule:** Do not silently discard rows with missing multiclass targets or assume that category names are consistent. Inspect the evidence first and decide how to handle any issues explicitly.

In [2]:
# Load the official splits and audit the multiclass target before preprocessing.

if not TRAIN_PATH.is_file() or not TEST_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset file not found. Training: {TRAIN_PATH}; Testing: {TEST_PATH}"
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("TRAINING SET")
print("=" * 60)
print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

print("\nTESTING SET")
print("=" * 60)
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

required_columns = (
    set(REDUCED_NUMERICAL_FEATURES)
    | set(CATEGORICAL_FEATURES)
    | {TARGET_COLUMN}
)

for split_name, df in [("training", train_df), ("testing", test_df)]:
    missing_columns = sorted(required_columns - set(df.columns))
    print(f"\n{split_name.title()} split")
    print("Missing required columns:", missing_columns)
    print(f"Missing '{TARGET_COLUMN}' values:", df[TARGET_COLUMN].isna().sum()
          if TARGET_COLUMN in df.columns else "target column absent")

    if TARGET_COLUMN in df.columns:
        print("Unique target values:", sorted(
            df[TARGET_COLUMN].dropna().astype(str).unique().tolist()
        ))
        print("\nTarget distribution (including missing values):")
        print(df[TARGET_COLUMN].value_counts(dropna=False))

print("\nBinary label distribution — audit only")
if "label" in train_df.columns:
    print(train_df["label"].value_counts(dropna=False))

TRAINING SET
Shape: (82332, 45)
Columns: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

TESTING SET
Shape: (175341, 45)
Columns: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is

## 4. Experimental Protocol and Leakage Controls

This experiment compares multiclass intrusion detection classifiers using the complete official UNSW-NB15 training dataset. All 82,332 training records will be retained.

- **Target:** `attack_cat` — ten-class classification.
- **Training data:** the complete official training split, containing 82,332 records.
- **Final evaluation data:** the complete official test split, containing 175,341 records.
- **Features:** the existing Variant E feature set, comprising 26 numerical and 3 categorical features.
- **Excluded columns:** `attack_cat`, `label`, and `id`.
- **Preprocessing:** fit the Variant E preprocessor on the full training features and apply the fitted transformer to the official test features.
- **Candidate comparison:** train each candidate on the full training set. Record training metrics and computational costs as diagnostics, not as evidence of generalisation.
- **Final evaluation:** evaluate the selected candidate on the official test set after model selection. Test results will not be used to tune or select candidates.

**Evaluation considerations**

Macro-F1 will be the primary evaluation metric because it gives equal weight to each attack category. Weighted-F1, accuracy, and per-class precision, recall, and F1 will provide additional context.

The `Worms` class contains only 44 training records. All available examples will be retained, but performance estimates for this minority class may be unstable.

**Leakage controls**

The target and identifier columns will not enter the feature matrix. The preprocessor will be fitted only on training data. The official test set will remain separate from model fitting and model selection until final evaluation.

Training performance may be optimistic, particularly for high-capacity models. The final test results will provide the principal assessment of generalisation to the supplied official test split.

In [4]:
# Use the complete official training split for model development.
# The official test split remains untouched until final evaluation.

feature_columns = REDUCED_NUMERICAL_FEATURES + CATEGORICAL_FEATURES

X_train_full = train_df[feature_columns].copy()
y_train_full = train_df[TARGET_COLUMN].copy()

X_test_official = test_df[feature_columns].copy()
y_test_official = test_df[TARGET_COLUMN].copy()

# Prevent target leakage and identifier leakage.
for excluded_column in [TARGET_COLUMN, "label", "id"]:
    assert excluded_column not in X_train_full.columns
    assert excluded_column not in X_test_official.columns

assert not y_train_full.isna().any()
assert not y_test_official.isna().any()
assert X_train_full.columns.tolist() == X_test_official.columns.tolist()
assert set(y_train_full.unique()) == set(y_test_official.unique())

print("Full training feature shape:", X_train_full.shape)
print("Official test feature shape:", X_test_official.shape)

print("\nFull training class distribution:")
display(y_train_full.value_counts().sort_index().rename("count").to_frame())

print("\nOfficial test class distribution:")
display(y_test_official.value_counts().sort_index().rename("count").to_frame())

print("\nAll classes are represented in both splits.")
print("No target or identifier columns are included in the features.")
print("Official test data has not been used for fitting.")

Full training feature shape: (82332, 29)
Official test feature shape: (175341, 29)

Full training class distribution:


,count
attack_cat,
Analysis,677
Backdoor,583
DoS,4089
Exploits,11132
Fuzzers,6062
Generic,18871
Normal,37000
Reconnaissance,3496
Shellcode,378



Official test class distribution:


,count
attack_cat,
Analysis,2000
Backdoor,1746
DoS,12264
Exploits,33393
Fuzzers,18184
Generic,40000
Normal,56000
Reconnaissance,10491
Shellcode,1133



All classes are represented in both splits.
No target or identifier columns are included in the features.
Official test data has not been used for fitting.


## 5. Feature Preprocessing

This experiment reuses the existing AEGES-Q Variant E preprocessing implementation to maintain consistency with the binary classifier's feature engineering.

The preprocessor applies median imputation and standardisation to numerical features, and most-frequent imputation and one-hot encoding to categorical features. Unknown categorical values in the test set are handled using the existing encoder configuration.

The transformer is fitted exclusively on the official training features. The fitted transformer is then applied to both training and test features. Target labels and identifier columns remain outside the feature matrices.

The resulting transformed matrices will be checked for compatible dimensions and finite numerical values before classifier training.

In [5]:
# Fit the existing preprocessor on all official training rows.
# Reuse this fitted transformer for every candidate classifier.

preprocessor = build_variant_e_preprocessor()

X_train_processed = preprocessor.fit_transform(X_train_full)
X_test_processed = preprocessor.transform(X_test_official)

print("Raw training shape:", X_train_full.shape)
print("Raw test shape:", X_test_official.shape)

print("\nProcessed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

print("\nPreprocessor fitted on training data only.")
print("Train/test processed feature dimensions match:",
      X_train_processed.shape[1] == X_test_processed.shape[1])

# Check transformed values without densifying sparse matrices.
from scipy import sparse

def all_finite(matrix):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return bool(np.isfinite(values).all())

print("Training features contain only finite values:",
      all_finite(X_train_processed))
print("Test features contain only finite values:",
      all_finite(X_test_processed))

assert X_train_processed.shape[0] == len(y_train_full)
assert X_test_processed.shape[0] == len(y_test_official)
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert all_finite(X_train_processed)
assert all_finite(X_test_processed)

Raw training shape: (82332, 29)
Raw test shape: (175341, 29)

Processed training shape: (82332, 177)
Processed test shape: (175341, 177)

Preprocessor fitted on training data only.
Train/test processed feature dimensions match: True
Training features contain only finite values: True
Test features contain only finite values: True


## 6. Candidate Classifiers and Comparison Strategy

Four classifiers are included in the initial controlled comparison:

1. **Dummy classifier:** a majority-class baseline that establishes a minimum reference point.
2. **Logistic regression:** a linear multiclass baseline.
3. **Random forest:** a tree-ensemble baseline consistent with the existing AEGES-Q classical model.
4. **Extra trees:** a randomized tree ensemble for comparison with random forest.

All candidates receive the same preprocessed training matrix and use the same training labels. Each candidate is fitted on the complete official training split.

Training duration, accuracy, macro-F1, and weighted-F1 will be recorded as diagnostic measurements. Training metrics are not a reliable estimate of generalisation, especially for high-capacity tree ensembles. Candidate definitions and selection criteria should remain fixed before final test evaluation.

The initial comparison uses fixed random seeds where supported. No hyperparameter search or resampling is performed in this stage. Any later tuning must be documented as a separate experiment.

In [6]:
# Define candidates before training.
# All models use the same preprocessed training matrix.

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

candidates = {
    "dummy_most_frequent": DummyClassifier(
        strategy="most_frequent"
    ),
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

print("Candidate models:")
for name, model in candidates.items():
    print(f"- {name}: {model}")

print(f"\nTraining rows per candidate: {len(y_train_full):,}")
print(f"Processed features per row: {X_train_processed.shape[1]}")
print(f"Number of classes: {y_train_full.nunique()}")

Candidate models:
- dummy_most_frequent: DummyClassifier(strategy='most_frequent')
- logistic_regression: LogisticRegression(max_iter=1000, random_state=42)
- random_forest: RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
- extra_trees: ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=42)

Training rows per candidate: 82,332
Processed features per row: 177
Number of classes: 10


## 7. Full-Training Candidate Fit and Diagnostic Metrics

Each candidate will be trained on the complete official training dataset. The experiment records elapsed fit time, training accuracy, macro-F1, and weighted-F1 in a single results table.

These training metrics are diagnostic only. They indicate how well each fitted model reproduces the training labels, not how well it will classify unseen network traffic. The official test set is not used in this cell.

The fitted estimators and their measurements will be retained for subsequent analysis. Any candidate that fails to fit will be recorded with its error rather than silently omitted.

In [7]:
# Train every candidate on all official training rows.
# This cell deliberately does not evaluate on the official test set.

training_results = []
fitted_models = {}
training_errors = {}

for name, model in candidates.items():
    print(f"\nTraining: {name}")
    start_time = time.perf_counter()

    try:
        model.fit(X_train_processed, y_train_full)
        fit_seconds = time.perf_counter() - start_time

        y_train_pred = model.predict(X_train_processed)

        result = {
            "model": name,
            "status": "success",
            "fit_seconds": round(fit_seconds, 3),
            "train_accuracy": accuracy_score(y_train_full, y_train_pred),
            "train_macro_f1": f1_score(
                y_train_full, y_train_pred,
                average="macro", zero_division=0
            ),
            "train_weighted_f1": f1_score(
                y_train_full, y_train_pred,
                average="weighted", zero_division=0
            ),
            "error": None,
        }

        fitted_models[name] = model
        training_results.append(result)

        print(f"Fit time: {fit_seconds:.2f} seconds")
        print(f"Training accuracy: {result['train_accuracy']:.4f}")
        print(f"Training macro-F1: {result['train_macro_f1']:.4f}")
        print(f"Training weighted-F1: {result['train_weighted_f1']:.4f}")

    except Exception as exc:
        fit_seconds = time.perf_counter() - start_time
        error_message = f"{type(exc).__name__}: {exc}"
        training_errors[name] = error_message

        training_results.append({
            "model": name,
            "status": "failed",
            "fit_seconds": round(fit_seconds, 3),
            "train_accuracy": np.nan,
            "train_macro_f1": np.nan,
            "train_weighted_f1": np.nan,
            "error": error_message,
        })

        print(f"FAILED after {fit_seconds:.2f} seconds: {error_message}")

training_results_df = pd.DataFrame(training_results)

print("\nTraining comparison")
display(
    training_results_df.sort_values(
        by="train_macro_f1",
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)
)


Training: dummy_most_frequent
Fit time: 0.07 seconds
Training accuracy: 0.4494
Training macro-F1: 0.0620
Training weighted-F1: 0.2787

Training: logistic_regression
Fit time: 30.61 seconds
Training accuracy: 0.7944
Training macro-F1: 0.3711
Training weighted-F1: 0.7676

Training: random_forest
Fit time: 107.76 seconds
Training accuracy: 0.9325
Training macro-F1: 0.7580
Training weighted-F1: 0.9297

Training: extra_trees
Fit time: 141.47 seconds
Training accuracy: 0.9325
Training macro-F1: 0.7616
Training weighted-F1: 0.9304

Training comparison


,model,status,fit_seconds,train_accuracy,train_macro_f1,train_weighted_f1,error
0,extra_trees,success,141.466,0.932505,0.761633,0.930397,None
1,random_forest,success,107.761,0.932505,0.757954,0.929733,None
2,logistic_regression,success,30.607,0.794442,0.371065,0.767617,None
3,dummy_most_frequent,success,0.072,0.449400,0.062012,0.278681,None


## 8. Training Diagnostics and Provisional Candidate

The initial full-training comparison shows that Extra Trees and Random Forest achieve similar training performance, with Extra Trees obtaining the higher macro-F1 and Random Forest requiring less fit time. Logistic regression provides a weaker linear reference, while the dummy classifier establishes a majority-class baseline.

These results do not establish which model generalises best. Training performance can be optimistic, and the difference between the two tree ensembles is small. Extra Trees is therefore designated as the provisional candidate for further diagnostic inspection, not as a confirmed final model.

The next step examines class-wise training metrics and the fitted models' behaviour. These diagnostics help identify classes that are difficult to learn, but they cannot substitute for evaluation on unseen data.

In [8]:
# Inspect per-class training performance for the two tree ensembles.
# These are diagnostics, not generalisation estimates.

tree_models = [
    name for name in ["extra_trees", "random_forest"]
    if name in fitted_models
]

class_diagnostic_frames = []

for name in tree_models:
    y_pred = fitted_models[name].predict(X_train_processed)

    report = classification_report(
        y_train_full,
        y_pred,
        labels=sorted(y_train_full.unique()),
        output_dict=True,
        zero_division=0,
    )

    class_frame = (
        pd.DataFrame(report)
        .T
        .loc[sorted(y_train_full.unique()), ["precision", "recall", "f1-score", "support"]]
        .reset_index(names="attack_cat")
    )
    class_frame.insert(0, "model", name)
    class_diagnostic_frames.append(class_frame)

class_training_diagnostics = pd.concat(
    class_diagnostic_frames,
    ignore_index=True,
)

display(
    class_training_diagnostics.sort_values(
        ["attack_cat", "model"]
    ).reset_index(drop=True)
)

,model,attack_cat,precision,recall,f1-score,support
0,extra_trees,Analysis,1.000000,0.085672,0.157823,677.0
1,random_forest,Analysis,1.000000,0.085672,0.157823,677.0
2,extra_trees,Backdoor,0.730337,0.111492,0.193452,583.0
3,random_forest,Backdoor,1.000000,0.097770,0.178125,583.0
4,extra_trees,DoS,0.570070,0.835657,0.677774,4089.0
5,random_forest,DoS,0.570828,0.829787,0.676368,4089.0
6,extra_trees,Exploits,0.777480,0.834890,0.805163,11132.0
7,random_forest,Exploits,0.844284,0.798778,0.820901,11132.0
8,extra_trees,Fuzzers,0.976466,0.766579,0.858886,6062.0
9,random_forest,Fuzzers,0.816680,0.831904,0.824222,6062.0


In [9]:
# Check which boosting implementations are available in the current environment.

import importlib.util

boosting_packages = {
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "catboost": "catboost",
}

for package, module_name in boosting_packages.items():
    available = importlib.util.find_spec(module_name) is not None
    print(f"{package}: {'available' if available else 'not installed'}")

print("\nscikit-learn HistGradientBoostingClassifier is available through sklearn.")

xgboost: not installed
lightgbm: not installed
catboost: not installed

scikit-learn HistGradientBoostingClassifier is available through sklearn.


## 9. Extended Model Comparison: Histogram Gradient Boosting

This experiment extends the initial comparison with a gradient-boosted tree classifier. Unlike Random Forest and Extra Trees, which combine independently trained trees, gradient boosting builds trees sequentially to improve the current ensemble.

The initial model will use scikit-learn's `HistGradientBoostingClassifier`. Its fit time, training macro-F1, weighted-F1, and accuracy will be recorded using the same full training dataset and transformed features as the baseline experiment.

Training metrics remain diagnostic only. This experiment does not claim improved generalisation unless supported by evaluation on unseen data. The official test set remains excluded from candidate fitting and model selection.

Before fitting, the transformed matrix representation will be checked to ensure compatibility with the estimator and to avoid unnecessary memory usage.

In [10]:
# Inspect the processed matrix representation before selecting a compatible estimator.

from scipy import sparse

print("Training matrix type:", type(X_train_processed).__name__)
print("Test matrix type:", type(X_test_processed).__name__)
print("Training matrix sparse:", sparse.issparse(X_train_processed))
print("Test matrix sparse:", sparse.issparse(X_test_processed))

if sparse.issparse(X_train_processed):
    estimated_dense_gb = (
        X_train_processed.shape[0]
        * X_train_processed.shape[1]
        * np.dtype(np.float64).itemsize
        / (1024 ** 3)
    )
    print(f"Estimated dense training size (float64): {estimated_dense_gb:.3f} GiB")
else:
    print("Training matrix dtype:", X_train_processed.dtype)
    print(
        "Training matrix memory (MiB):",
        round(X_train_processed.nbytes / (1024 ** 2), 2)
    )

Training matrix type: csr_matrix
Test matrix type: csr_matrix
Training matrix sparse: True
Test matrix sparse: True
Estimated dense training size (float64): 0.109 GiB


## 10. Histogram Gradient Boosting: Initial Fit

The preprocessed feature matrices are stored as CSR sparse matrices. For this experiment, they will be converted to dense arrays because the estimated training matrix size is modest for the available environment.

A baseline `HistGradientBoostingClassifier` will be fitted on the complete official training dataset. Its training time and training metrics will be compared with the initial tree ensembles.

This is an additional candidate, not an assumed improvement. Training performance will be treated as diagnostic; final model selection must not rely on training scores alone.

In [11]:
# Convert only for HistGradientBoosting; keep the shared sparse matrices unchanged.

from sklearn.ensemble import HistGradientBoostingClassifier

X_train_dense = X_train_processed.toarray()
X_test_dense = X_test_processed.toarray()

print("Dense training shape:", X_train_dense.shape)
print("Dense test shape:", X_test_dense.shape)
print("Dense training dtype:", X_train_dense.dtype)
print(
    "Dense training memory (MiB):",
    round(X_train_dense.nbytes / (1024 ** 2), 2)
)

hist_gradient_boosting = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=RANDOM_STATE,
)

start_time = time.perf_counter()
hist_gradient_boosting.fit(X_train_dense, y_train_full)
hist_fit_seconds = time.perf_counter() - start_time

hist_train_pred = hist_gradient_boosting.predict(X_train_dense)

hist_result = {
    "model": "hist_gradient_boosting",
    "status": "success",
    "fit_seconds": round(hist_fit_seconds, 3),
    "train_accuracy": accuracy_score(y_train_full, hist_train_pred),
    "train_macro_f1": f1_score(
        y_train_full, hist_train_pred,
        average="macro", zero_division=0
    ),
    "train_weighted_f1": f1_score(
        y_train_full, hist_train_pred,
        average="weighted", zero_division=0
    ),
}

print("\nHistGradientBoosting results")
for metric, value in hist_result.items():
    print(f"{metric}: {value}")

Dense training shape: (82332, 177)
Dense test shape: (175341, 177)
Dense training dtype: float64
Dense training memory (MiB): 111.18

HistGradientBoosting results
model: hist_gradient_boosting
status: success
fit_seconds: 84.354
train_accuracy: 0.8914881212651217
train_macro_f1: 0.6915453124531149
train_weighted_f1: 0.8857314215525621


## 11. Tree-Ensemble Capacity Experiment

This experiment investigates how tree complexity affects Random Forest and Extra Trees on the complete official training dataset.

The configurations vary tree depth and minimum leaf size while keeping the number of trees and random seed fixed. The aim is to measure changes in fit time and training performance, not to claim that the best training score will generalise best.

The experiment reuses the already-fitted Variant E preprocessor. The official test set remains excluded from fitting and configuration selection.

In [12]:
# Small, controlled capacity sweep on the complete training dataset.
# These training metrics are diagnostic only.

from sklearn.base import clone

tree_configs = [
    {
        "name": "rf_depth_none_leaf1",
        "family": "random_forest",
        "params": {"max_depth": None, "min_samples_leaf": 1},
    },
    {
        "name": "rf_depth_30_leaf1",
        "family": "random_forest",
        "params": {"max_depth": 30, "min_samples_leaf": 1},
    },
    {
        "name": "rf_depth_none_leaf2",
        "family": "random_forest",
        "params": {"max_depth": None, "min_samples_leaf": 2},
    },
    {
        "name": "et_depth_none_leaf1",
        "family": "extra_trees",
        "params": {"max_depth": None, "min_samples_leaf": 1},
    },
    {
        "name": "et_depth_30_leaf1",
        "family": "extra_trees",
        "params": {"max_depth": 30, "min_samples_leaf": 1},
    },
    {
        "name": "et_depth_none_leaf2",
        "family": "extra_trees",
        "params": {"max_depth": None, "min_samples_leaf": 2},
    },
]

base_estimators = {
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

tuning_results = []
tuned_models = {}

for config in tree_configs:
    name = config["name"]
    model = clone(base_estimators[config["family"]]).set_params(**config["params"])

    print(f"\nTraining: {name}")
    start_time = time.perf_counter()

    model.fit(X_train_processed, y_train_full)
    fit_seconds = time.perf_counter() - start_time

    train_pred = model.predict(X_train_processed)

    result = {
        "model": name,
        "family": config["family"],
        "fit_seconds": round(fit_seconds, 3),
        "train_accuracy": accuracy_score(y_train_full, train_pred),
        "train_macro_f1": f1_score(
            y_train_full, train_pred,
            average="macro", zero_division=0
        ),
        "train_weighted_f1": f1_score(
            y_train_full, train_pred,
            average="weighted", zero_division=0
        ),
        **config["params"],
    }

    tuning_results.append(result)
    tuned_models[name] = model

    print(
        f"Fit: {fit_seconds:.1f}s | "
        f"Macro-F1: {result['train_macro_f1']:.4f} | "
        f"Accuracy: {result['train_accuracy']:.4f}"
    )

tree_tuning_df = pd.DataFrame(tuning_results)

display(
    tree_tuning_df.sort_values(
        "train_macro_f1", ascending=False
    ).reset_index(drop=True)
)


Training: rf_depth_none_leaf1
Fit: 276.6s | Macro-F1: 0.7580 | Accuracy: 0.9325

Training: rf_depth_30_leaf1
Fit: 160.7s | Macro-F1: 0.7504 | Accuracy: 0.9277

Training: rf_depth_none_leaf2
Fit: 69.0s | Macro-F1: 0.6215 | Accuracy: 0.9038

Training: et_depth_none_leaf1
Fit: 146.6s | Macro-F1: 0.7616 | Accuracy: 0.9325

Training: et_depth_30_leaf1
Fit: 78.8s | Macro-F1: 0.6004 | Accuracy: 0.8743

Training: et_depth_none_leaf2
Fit: 53.1s | Macro-F1: 0.4077 | Accuracy: 0.8255


,model,family,fit_seconds,train_accuracy,train_macro_f1,train_weighted_f1,max_depth,min_samples_leaf
0,et_depth_none_leaf1,extra_trees,146.572,0.932505,0.761633,0.930397,NaN,1
1,rf_depth_none_leaf1,random_forest,276.643,0.932505,0.757954,0.929733,NaN,1
2,rf_depth_30_leaf1,random_forest,160.693,0.927744,0.750367,0.924486,30.0,1
3,rf_depth_none_leaf2,random_forest,68.997,0.903768,0.621523,0.898682,NaN,2
4,et_depth_30_leaf1,extra_trees,78.844,0.874338,0.600377,0.861039,30.0,1
5,et_depth_none_leaf2,extra_trees,53.082,0.825463,0.407716,0.798009,NaN,2
